# Chapter 7 — CNNs
**Track:** ML from Scratch · Synthetic neighbourhood grid images

## Core Idea

A **Convolutional Neural Network** applies learned filters that slide across the input — sharing the same weights at every spatial position.

```
Dense (Ch.4):  64 × 128 + 128 = 8,320 params for first layer
CNN:           (3×3×1 + 1) × 8 = 80 params for equivalent first conv layer
```

Key ingredients: **Convolution → ReLU → Pooling**, repeated, then **Dense head**.

## Running Example

Synthetic **8×8 neighbourhood grid images** (greyscale):
- `label=0` (tidy): high mean brightness, low variance
- `label=1` (distressed): low mean brightness, high variance (patchy lots)

This mirrors real property-condition classification from aerial imagery — same task, fully self-contained without downloading external data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ── Synthetic image dataset ───────────────────────────────────────────────────
def make_neighbourhood_grids(n_samples=2000, grid_size=8, seed=42):
    """Create synthetic 8×8 greyscale neighbourhood grids.
    Shape: (N, 1, H, W) — channels-first convention (matches Keras channels_first).
    """
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n_samples // 2):
        # Tidy: bright patches, small noise
        grid = rng.normal(0.75, 0.10, (grid_size, grid_size)).clip(0, 1)
        X.append(grid[np.newaxis]);  y.append(0)
        # Distressed: darker, patchy (some bright, some very dark)
        grid = rng.normal(0.35, 0.30, (grid_size, grid_size)).clip(0, 1)
        X.append(grid[np.newaxis]);  y.append(1)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_img, y_img = make_neighbourhood_grids(n_samples=2000)
X_tr, X_te, y_tr, y_te = train_test_split(X_img, y_img, test_size=0.2, random_state=42)

print(f"Dataset: {X_img.shape}  classes: {np.bincount(y_img)}")
print(f"Pixel range: [{X_img.min():.3f}, {X_img.max():.3f}]  (already in [0,1])")

In [ ]:
# Visualise sample grids from each class
fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
for cls, label in [(0, 'Tidy'), (1, 'Distressed')]:
    samples = X_img[y_img == cls][:8]
    for j, ax in enumerate(axes[cls]):
        ax.imshow(samples[j, 0], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
    axes[cls][0].set_title(f'{label} (label={cls})', loc='left', fontsize=10)

plt.suptitle('Synthetic Neighbourhood Grids (8×8)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Manual 2D Convolution (NumPy)

$$
(\mathbf{X} * \mathbf{K})_{i,j} = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} \mathbf{X}_{i+u,\,j+v} \cdot \mathbf{K}_{u,v}
$$

Output size (no padding, stride 1): $H_\text{out} = H - k + 1$

In [ ]:
def conv2d_numpy(x, kernel, stride=1, padding=0):
    """Manual 2D cross-correlation (single channel).
    x:      (H, W)
    kernel: (k, k)
    Returns (H_out, W_out).
    """
    k = kernel.shape[0]
    if padding:
        x = np.pad(x, padding, mode='constant')
    H, W = x.shape
    H_out = (H - k) // stride + 1
    W_out = (W - k) // stride + 1
    out = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            out[i, j] = (x[i*stride:i*stride+k, j*stride:j*stride+k] * kernel).sum()
    return out

def maxpool2d_numpy(x, pool_size=2):
    """Non-overlapping max pooling."""
    H, W = x.shape
    H_out, W_out = H // pool_size, W // pool_size
    out = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            out[i, j] = x[i*pool_size:(i+1)*pool_size,
                          j*pool_size:(j+1)*pool_size].max()
    return out

# ── Demo on one tidy and one distressed grid ──────────────────────────────────
tidy_img       = X_img[y_img == 0][0, 0]    # (8, 8)
distressed_img = X_img[y_img == 1][0, 0]    # (8, 8)

# Hand-crafted edge-detection kernel (Sobel-like horizontal edges)
edge_kernel = np.array([[-1, -1, -1],
                         [ 0,  0,  0],
                         [ 1,  1,  1]], dtype=np.float32)

tidy_edge  = conv2d_numpy(tidy_img,       edge_kernel)
dist_edge  = conv2d_numpy(distressed_img, edge_kernel)

tidy_pool  = maxpool2d_numpy(np.maximum(0, tidy_edge),  pool_size=2)  # ReLU then pool
dist_pool  = maxpool2d_numpy(np.maximum(0, dist_edge),  pool_size=2)

fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for row, (raw, edge, pool, label) in enumerate([
    (tidy_img,       tidy_edge,  tidy_pool,  'Tidy'),
    (distressed_img, dist_edge,  dist_pool,  'Distressed'),
]):
    axes[row][0].imshow(raw,  cmap='gray', vmin=0, vmax=1)
    axes[row][0].set_title(f'{label} — Input (8×8)')
    axes[row][1].imshow(edge, cmap='RdBu_r')
    axes[row][1].set_title(f'{label} — After Conv (6×6)')
    axes[row][2].imshow(pool, cmap='RdBu_r')
    axes[row][2].set_title(f'{label} — After MaxPool (3×3)')
    for ax in axes[row]: ax.axis('off')

plt.suptitle('Manual Conv → ReLU → MaxPool', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Input: {tidy_img.shape}  → Conv: {tidy_edge.shape}  → Pool: {tidy_pool.shape}")

## Filter Visualisation — What Different Kernels Detect

Different hand-crafted kernels detect different patterns. CNNs learn these kernels automatically from data.

In [ ]:
kernels = {
    'Horizontal edges': np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32),
    'Vertical edges':   np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32),
    'Sharpen':          np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
    'Blur (mean)':      np.ones((3,3), dtype=np.float32) / 9,
}

sample = tidy_img   # (8, 8)
fig, axes = plt.subplots(1, len(kernels) + 1, figsize=(14, 3))
axes[0].imshow(sample, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input (8×8)')
axes[0].axis('off')

for ax, (name, k) in zip(axes[1:], kernels.items()):
    out = conv2d_numpy(sample, k)
    ax.imshow(out, cmap='RdBu_r')
    ax.set_title(f'{name}\n{out.shape}')
    ax.axis('off')

plt.suptitle('What Different Filters Detect (hand-crafted kernels)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Parameter Count: Dense vs CNN

Why do CNNs need far fewer parameters for the same input?

In [ ]:
# Input: 1×8×8 = 64 pixels
input_pixels = 1 * 8 * 8

# Dense first layer: 64 inputs → 128 units
dense_params = input_pixels * 128 + 128

# Conv first layer: 3×3 kernel, 1 input channel, 8 filters
conv_params = (3 * 3 * 1 + 1) * 8

print(f"Dense (64 → 128):           {dense_params:,} parameters")
print(f"Conv2D (3×3, 1→8 filters):  {conv_params:,} parameters")
print(f"Reduction factor:           {dense_params / conv_params:.0f}×")

# Scale up to realistic image size: 224×224 RGB
large_input = 3 * 224 * 224
dense_large = large_input * 128 + 128
conv_large  = (3 * 3 * 3 + 1) * 32   # 3×3 kernel, 3 RGB channels → 32 filters
print(f"\nFor 224×224 RGB image:")
print(f"Dense (150,528 → 128):      {dense_large:,} parameters")
print(f"Conv2D (3×3, 3→32 filters): {conv_large:,} parameters")
print(f"Reduction factor:           {dense_large / conv_large:,.0f}×")

## CNN Classifier with Keras

Build and train a CNN on the synthetic neighbourhood grids.
Architecture: `Input(1,8,8) → Conv(8,3×3) → MaxPool → Conv(16,3×3) → Flatten → Dense(32) → Dense(1, sigmoid)`

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print(f"TensorFlow {tf.__version__} available")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not installed — CNN training cells will be skipped.")
    print("Install with: pip install tensorflow")

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(42)

    # Keras uses channels-last by default: (N, H, W, C)
    X_tr_cl = X_tr.transpose(0, 2, 3, 1)   # (N, 1, 8, 8) → (N, 8, 8, 1)
    X_te_cl = X_te.transpose(0, 2, 3, 1)

    cnn = keras.Sequential([
        layers.Input(shape=(8, 8, 1)),
        layers.Conv2D(8, 3, activation='relu', padding='valid'),   # → (6,6,8)
        layers.MaxPooling2D(2),                                     # → (3,3,8)
        layers.Conv2D(16, 3, activation='relu', padding='valid'),  # → (1,1,16)
        layers.Flatten(),                                            # → 16
        layers.Dense(32, activation='relu'),
        layers.Dense(1,  activation='sigmoid'),
    ], name='PropertyConditionCNN')

    cnn.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )
    cnn.summary()
else:
    print("Skipped — TensorFlow not available.")

In [ ]:
if TF_AVAILABLE:
    history = cnn.fit(
        X_tr_cl, y_tr,
        epochs=40,
        batch_size=64,
        validation_split=0.15,
        verbose=0,
    )

    y_pred_prob = cnn.predict(X_te_cl, verbose=0).ravel()
    y_pred      = (y_pred_prob > 0.5).astype(int)
    acc         = accuracy_score(y_te, y_pred)
    print(f"Test accuracy: {acc:.4f}")
    print(classification_report(y_te, y_pred, target_names=['Tidy', 'Distressed']))

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['loss'],     label='Train loss',     color='steelblue')
    axes[0].plot(history.history['val_loss'], label='Val loss',   color='coral', linestyle='--')
    axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history['accuracy'],     label='Train acc', color='steelblue')
    axes[1].plot(history.history['val_accuracy'], label='Val acc',   color='coral', linestyle='--')
    axes[1].set_title('Accuracy Curve'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('CNN Training — Neighbourhood Condition Classifier', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — TensorFlow not available.")

## Learned Filter Visualisation

After training, the first conv layer's 8 filters have been learned from data. Visualise them and their responses.

In [ ]:
if TF_AVAILABLE:
    # First conv layer weights: shape (k_h, k_w, in_channels, n_filters) = (3,3,1,8)
    conv1_weights = cnn.layers[0].get_weights()[0]   # (3,3,1,8)

    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    for f in range(8):
        filt = conv1_weights[:, :, 0, f]   # (3,3)
        axes[0][f].imshow(filt, cmap='RdBu_r')
        axes[0][f].set_title(f'Filter {f+1}', fontsize=8)
        axes[0][f].axis('off')

        # Show response on a tidy sample
        resp = conv2d_numpy(tidy_img, filt)
        axes[1][f].imshow(np.maximum(0, resp), cmap='hot')
        axes[1][f].set_title('Response', fontsize=8)
        axes[1][f].axis('off')

    axes[0][0].set_ylabel('Learned filters', fontsize=9)
    axes[1][0].set_ylabel('Activations', fontsize=9)
    plt.suptitle('Learned Conv1 Filters and Their Responses (tidy sample)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — TensorFlow not available.")

## Filter Depth Sweep — How Many Filters?

Vary the number of filters in the first conv layer and track test accuracy.

In [ ]:
if TF_AVAILABLE:
    filter_counts = [2, 4, 8, 16, 32]
    filter_accs = []

    for n_filters in filter_counts:
        m = keras.Sequential([
            layers.Input(shape=(8, 8, 1)),
            layers.Conv2D(n_filters, 3, activation='relu', padding='valid'),
            layers.MaxPooling2D(2),
            layers.Flatten(),
            layers.Dense(16, activation='relu'),
            layers.Dense(1,  activation='sigmoid'),
        ])
        m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        m.fit(X_tr_cl, y_tr, epochs=30, batch_size=64, verbose=0, validation_split=0.1)
        _, acc = m.evaluate(X_te_cl, y_te, verbose=0)
        filter_accs.append(acc)
        print(f"n_filters={n_filters:3d}  test_acc={acc:.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(filter_counts, filter_accs, marker='o', color='steelblue')
    ax.set_xlabel('Number of filters (first conv layer)')
    ax.set_ylabel('Test Accuracy')
    ax.set_title('Filter Count Sweep — Property Condition CNN')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — TensorFlow not available.")

## Trap 1 — Kernel Size Too Large for Small Inputs

A 5×5 kernel on an 8×8 input (no padding, stride 1) produces a 4×4 output. After MaxPool(2×2) → 2×2. One more conv(3×3): output is 0×0 — the network crashes.

In [ ]:
# Trace spatial size through the network for increasing kernel sizes
import math

def output_size(h, k, s=1, p=0):
    return math.floor((h + 2*p - k) / s) + 1

print(f"{'Kernel':>8} {'After Conv1':>12} {'After Pool':>12} {'After Conv2':>12} {'Valid?':>8}")
print("-" * 58)
for k in [3, 5, 7]:
    h_after_conv1 = output_size(8, k)
    h_after_pool  = h_after_conv1 // 2 if h_after_conv1 >= 2 else 0
    h_after_conv2 = output_size(h_after_pool, 3) if h_after_pool >= 3 else 0
    valid         = h_after_conv2 > 0
    print(f"{k:>6}×{k:<1}   {h_after_conv1:>6}×{h_after_conv1:<4}  "
          f"{h_after_pool:>6}×{h_after_pool:<4}  "
          f"{h_after_conv2:>6}×{h_after_conv2:<4}  "
          f"{'✅' if valid else '❌ CRASH':>8}")

print("\nRule: always verify H_out = floor((H + 2p - k) / s) + 1 before building architecture.")

## Trap 2 — Unscaled Pixel Values

If pixel values are in [0, 255] instead of [0, 1], the first conv layer receives activations ~100× larger than expected. Gradients from the first layer are correspondingly larger → unstable training.

In [ ]:
# Simulate unscaled input: multiply by 255
X_unscaled = X_img * 255.0

# Compare activation magnitudes after one conv with random filter
rng = np.random.default_rng(42)
rand_filter = rng.normal(0, 1, (3, 3)).astype(np.float32)

sample_scaled   = X_img[0, 0]         # [0, 1]
sample_unscaled = X_unscaled[0, 0]    # [0, 255]

act_scaled   = conv2d_numpy(sample_scaled,   rand_filter)
act_unscaled = conv2d_numpy(sample_unscaled, rand_filter)

print(f"Scaled input [0,1]   — activation range: [{act_scaled.min():.2f}, {act_scaled.max():.2f}]")
print(f"Unscaled [0,255]     — activation range: [{act_unscaled.min():.2f}, {act_unscaled.max():.2f}]")
print(f"Magnitude ratio: {np.abs(act_unscaled).mean() / np.abs(act_scaled).mean():.1f}×")
print("\n→ Unscaled activations are ~255× larger → exploding gradients from step 0.")

## Exercises

**Exercise 1 — Global Average Pooling vs Flatten**  
Replace the `Flatten()` layer with `GlobalAveragePooling2D()` in the Keras model. Compare parameter counts and test accuracy. When would GAP be preferable?

**Exercise 2 — Receptive field calculator**  
Implement a function `receptive_field(n_layers, k)` that computes $1 + n \times (k-1)$. Plot the receptive field for `k=3` and `n_layers` from 1 to 10. At what depth does the receptive field exceed the 8×8 input?

**Exercise 3 — Custom edge detector**  
Generate a batch of synthetic grids where the tidy class has strong horizontal banding and the distressed class has random noise. Design a hand-crafted horizontal-edge kernel that achieves > 80% accuracy using only threshold classification on the mean filter response (no neural network needed). What does this tell you about the role of prior knowledge vs learned features?

In [ ]:
# Exercise 1 scaffold — GAP vs Flatten
if TF_AVAILABLE:
    cnn_gap = keras.Sequential([
        layers.Input(shape=(8, 8, 1)),
        layers.Conv2D(8, 3, activation='relu', padding='valid'),
        layers.MaxPooling2D(2),
        layers.Conv2D(16, 3, activation='relu', padding='valid'),
        layers.GlobalAveragePooling2D(),   # <-- instead of Flatten
        layers.Dense(16, activation='relu'),
        layers.Dense(1,  activation='sigmoid'),
    ])
    cnn_gap.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    print("GAP model summary:")
    cnn_gap.summary()
    # TODO: train and compare test accuracy to the Flatten model above

In [ ]:
# Exercise 2 scaffold — receptive field calculator
def receptive_field(n_layers, k, stride=1):
    """Receptive field after n_layers of conv with kernel k and given stride."""
    return 1 + n_layers * (k - 1)  # stride=1, padding=0 formula

layers_range = range(1, 11)
rf_3x3 = [receptive_field(n, k=3) for n in layers_range]
rf_5x5 = [receptive_field(n, k=5) for n in layers_range]

plt.figure(figsize=(8, 4))
plt.plot(layers_range, rf_3x3, marker='o', label='3×3 kernels')
plt.plot(layers_range, rf_5x5, marker='s', label='5×5 kernels')
plt.axhline(8, color='red', linestyle='--', label='Input size (8×8)')
plt.xlabel('Number of conv layers'); plt.ylabel('Receptive field (pixels)')
plt.title('Receptive Field Growth with Depth')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("3×3 — layers to exceed 8px input:", next(n for n in layers_range if rf_3x3[n-1] >= 8))

In [ ]:
# Exercise 3 scaffold — hand-crafted horizontal edge detector
def make_banded_grids(n_samples=500, grid_size=8, seed=42):
    """Tidy = horizontal bands; distressed = random noise."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n_samples // 2):
        g = np.zeros((grid_size, grid_size))
        for row in range(grid_size):
            g[row, :] = 0.8 if row % 2 == 0 else 0.2   # horizontal bands
        g += rng.normal(0, 0.05, g.shape)
        X.append(g.clip(0, 1)[np.newaxis]); y.append(0)

        X.append(rng.uniform(0, 1, (1, grid_size, grid_size))); y.append(1)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

X_band, y_band = make_banded_grids()
h_kernel = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32)

# TODO: compute mean filter response per image, threshold to classify, compute accuracy